# 05 — Cohort chung và chia thời gian

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Cùng ngày để so sánh công bằng

Bảng gốc có 4.671 phiên. SMA50 cần đủ 50 giá nên 49 hàng đầu chưa dùng được: còn **4.622 hàng** đủ chỉ báo giá. Sau ghép các nguồn và yêu cầu **60 phiên liên tiếp đủ 43 biến**, còn **1.560 hàng**, trong đó 1.559 hàng có nhãn.

- `date` đầu vào đầu tiên trong mẫu chung: **02/04/2019**.
- `target_date` tương ứng: **03/04/2019**, ngày đáp án được quan sát.
- Hàng đầu vào cuối **30/09/2025** chưa có nhãn phiên sau, giữ để dự báo.

Tại sao không dùng toàn bộ lịch sử giá từ 2007 cho mọi bảng? Vì các nguồn khác bắt đầu muộn và có khoảng trống. Nếu mỗi mô hình chấm trên ngày khác nhau thì khác biệt điểm có thể do giai đoạn thị trường, không chỉ do mô hình.

`session_index` lưu vị trí phiên trong bảng giá gốc. 60 hàng sau xóa thiếu chưa chắc là 60 phiên liền nhau; code kiểm tra cả chỉ số này để không ghép hai đoạn đứt thành cửa sổ GRU. Đây là yêu cầu đủ lịch sử đầu vào, không phải thêm 60 biến mới.

In [ ]:
from prepare import prepare,build_experiment_table,feature_groups,DEFAULT_INPUT,DEFAULT_OUTPUT
prepare(DEFAULT_INPUT,DEFAULT_OUTPUT)
base,common,_=build_experiment_table(DEFAULT_INPUT)
print({name:len(cols) for name,cols in feature_groups().items()})

## Train 85%, Test 15%; CV chỉ trong Train

Mỗi fold mở rộng phần học về phía trước, chừa 1 hàng trước đoạn kiểm định. Không xáo trộn thời gian.

In [ ]:
from train import _chronological_splits,cv_splits
data=common.dropna(subset=['next_log_return'])
masks,bounds=_chronological_splits(data)
print(json.dumps(bounds,indent=2))
display(pd.DataFrame([{'fold':i,'train_rows':len(a),'validation_rows':len(b),'gap_rows':b[0]-a[-1]-1} for i,(a,b) in enumerate(cv_splits(int(masks['train'].sum())),1)]))